# Explore Wikidata in Python

This notebook helps you interactively inspect what information exists in **Wikidata** for any term or entity.

## What is Wikidata?
Wikidata is an open, collaboratively edited knowledge base of structured data. It stores entities (items) such as people, places, organizations, concepts, and events, along with machine-readable statements about them.

## Key concepts
- **Entity / Item**: A thing in Wikidata, usually identified by IDs like `Q42` (items) and `P31` (properties).
- **Label**: Human-readable name of an entity (for example, "United States").
- **Description**: Short disambiguating text (for example, "country in North America").
- **Alias**: Alternative names, spellings, abbreviations, or synonyms.
- **Statement (Claim)**: A property-value assertion about an entity (for example, `P31` = "instance of").
- **Property**: The relation type used in statements (for example, `P17` = country, `P569` = date of birth).

## APIs used in this notebook
Wikidata can be explored through:
- **MediaWiki Action API** (`https://www.wikidata.org/w/api.php`)
  - `wbsearchentities`: search entities by term
  - `wbgetentities`: fetch detailed entity data
- **Wikidata Query Service** (`https://query.wikidata.org/sparql`) for SPARQL queries

The notebook focuses on readable outputs for exploration and beginner-friendly usage.

In [1]:
# Setup: import libraries, install missing packages if needed, and define API endpoints.

import sys
import subprocess
import importlib
import json

def ensure_package(package_name):
    """Install package via pip if not available."""
    try:
        importlib.import_module(package_name)
        print(f"Package '{package_name}' is available.")
    except ImportError:
        print(f"Installing missing package: {package_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

# Required third-party packages for this notebook.
for pkg in ["requests", "pandas"]:
    ensure_package(pkg)

import requests
import pandas as pd
from IPython.display import display, Markdown

# Base endpoints (official Wikidata-compatible patterns).
WIKIDATA_API_URL = "https://www.wikidata.org/w/api.php"
WDQS_URL = "https://query.wikidata.org/sparql"

# Good practice: identify your client in requests.
DEFAULT_HEADERS = {
    "User-Agent": "WikidataExplorerNotebook/1.0 (https://www.wikidata.org/)"
}

print("Setup complete.")
print("Wikidata API:", WIKIDATA_API_URL)
print("WDQS endpoint:", WDQS_URL)

Package 'requests' is available.
Package 'pandas' is available.
Setup complete.
Wikidata API: https://www.wikidata.org/w/api.php
WDQS endpoint: https://query.wikidata.org/sparql


## Search Wikidata Entities
Use `wbsearchentities` to find candidate entities for a word or phrase.

Endpoint pattern:
- `action=wbsearchentities`
- `search=<term>`
- `language=<lang>`
- `limit=<n>`

In [2]:
# Search functions: robust API call + DataFrame output.

def _safe_get_json(url, params=None, headers=None, timeout=30):
    """Safely call an HTTP JSON endpoint and return parsed JSON (or empty dict on failure)."""
    try:
        response = requests.get(url, params=params, headers=headers or DEFAULT_HEADERS, timeout=timeout)
        response.raise_for_status()
        return response.json()
    except requests.RequestException as exc:
        print(f"Request failed: {exc}")
        return {}
    except ValueError as exc:
        print(f"Invalid JSON response: {exc}")
        return {}

def _coerce_aliases_for_search(value):
    """Convert aliases from wbsearchentities result into a clean comma-separated string."""
    if value is None:
        return ""
    if isinstance(value, list):
        return ", ".join(str(v) for v in value if v is not None)
    if isinstance(value, str):
        return value
    return str(value)

def search_wikidata(term, language="en", limit=10):
    """
    Search Wikidata entities using wbsearchentities and return a clean DataFrame.

    Columns include: id, label, description, match_text, aliases, concepturi
    """
    if not isinstance(term, str) or not term.strip():
        print("Please provide a non-empty search term.")
        return pd.DataFrame(columns=["id", "label", "description", "match_text", "aliases", "concepturi"])

    params = {
        "action": "wbsearchentities",
        "format": "json",
        "language": language,
        "uselang": language,
        "search": term.strip(),
        "limit": int(limit),
    }

    data = _safe_get_json(WIKIDATA_API_URL, params=params)
    items = data.get("search", []) if isinstance(data, dict) else []

    rows = []
    for item in items:
        match = item.get("match") if isinstance(item, dict) else None
        match_text = ""
        if isinstance(match, dict):
            match_text = match.get("text", "")

        row = {
            "id": item.get("id", ""),
            "label": item.get("label", ""),
            "description": item.get("description", ""),
            "match_text": match_text,
            "aliases": _coerce_aliases_for_search(item.get("aliases")),
            "concepturi": item.get("concepturi", ""),
        }
        rows.append(row)

    df = pd.DataFrame(rows, columns=["id", "label", "description", "match_text", "aliases", "concepturi"])

    if df.empty:
        print(f"No search results for: {term!r}")
    return df

## Entity Detail Inspection
Use `wbgetentities` to retrieve details for a known entity ID (such as `Q30` or `Q312`).

Endpoint pattern:
- `action=wbgetentities`
- `ids=<entity_id>`
- `props=labels|descriptions|aliases|sitelinks|claims`

This section formats claims safely into readable text, with fallback handling for unexpected data shapes.

In [3]:
# Entity detail utilities and robust claim decoding.

def _extract_lang_value(lang_map, language="en"):
    """Get value for preferred language with fallback to first available."""
    if not isinstance(lang_map, dict) or not lang_map:
        return ""
    if language in lang_map and isinstance(lang_map[language], dict):
        return lang_map[language].get("value", "")
    first = next(iter(lang_map.values()), {})
    return first.get("value", "") if isinstance(first, dict) else ""

def _extract_aliases(alias_map, language="en"):
    """Get alias list for language with fallback to any language."""
    if not isinstance(alias_map, dict) or not alias_map:
        return []

    entries = alias_map.get(language)
    if not entries and alias_map:
        entries = next(iter(alias_map.values()), [])

    if not isinstance(entries, list):
        return []

    aliases = []
    for ent in entries:
        if isinstance(ent, dict):
            val = ent.get("value")
            if val:
                aliases.append(val)
    return aliases

def _decode_datavalue(datavalue):
    """Decode common Wikidata datavalue shapes into readable text."""
    if not isinstance(datavalue, dict):
        return str(datavalue)

    dtype = datavalue.get("type")
    value = datavalue.get("value")

    try:
        # wikibase-item / wikibase-property references
        if dtype == "wikibase-entityid" and isinstance(value, dict):
            entity_id = value.get("id")
            entity_type = value.get("entity-type", "entity")
            if entity_id:
                return f"{entity_id} ({entity_type})"
            if value.get("numeric-id") is not None:
                prefix = "Q" if entity_type == "item" else "P"
                return f"{prefix}{value['numeric-id']} ({entity_type})"
            return json.dumps(value, ensure_ascii=False)

        # plain strings
        if dtype == "string":
            return str(value)

        # monolingual text
        if dtype == "monolingualtext" and isinstance(value, dict):
            return f"{value.get('text', '')} [{value.get('language', '')}]"

        # time value
        if dtype == "time" and isinstance(value, dict):
            time_str = value.get("time", "")
            precision = value.get("precision", "")
            return f"{time_str} (precision={precision})"

        # quantity value
        if dtype == "quantity" and isinstance(value, dict):
            amount = value.get("amount", "")
            unit = value.get("unit", "")
            if isinstance(unit, str) and unit.startswith("http://www.wikidata.org/entity/"):
                unit = unit.rsplit("/", 1)[-1]
            return f"{amount} (unit={unit if unit else '1'})"

        # coordinates
        if dtype == "globecoordinate" and isinstance(value, dict):
            lat = value.get("latitude")
            lon = value.get("longitude")
            prec = value.get("precision")
            return f"lat={lat}, lon={lon}, precision={prec}"

        # Common fallback for other scalar types
        if isinstance(value, (str, int, float, bool)) or value is None:
            return str(value)

        # Safe fallback for unhandled complex types
        return json.dumps(value, ensure_ascii=False)

    except Exception:
        # Last-resort defensive fallback
        return str(datavalue)

def _decode_claim_value(claim):
    """Extract a readable value from one claim object."""
    if not isinstance(claim, dict):
        return "(invalid claim shape)"

    mainsnak = claim.get("mainsnak", {})
    if not isinstance(mainsnak, dict):
        return "(missing mainsnak)"

    snaktype = mainsnak.get("snaktype", "value")
    if snaktype != "value":
        return f"({snaktype})"

    datavalue = mainsnak.get("datavalue")
    if datavalue is None:
        return "(no datavalue)"

    return _decode_datavalue(datavalue)

def _get_entity_raw(entity_id, language="en"):
    """Fetch raw entity JSON from wbgetentities."""
    if not isinstance(entity_id, str) or not entity_id.strip():
        print("Please provide a valid entity ID like 'Q30'.")
        return None

    params = {
        "action": "wbgetentities",
        "format": "json",
        "ids": entity_id.strip(),
        "languages": language,
        "props": "labels|descriptions|aliases|sitelinks|claims",
    }

    data = _safe_get_json(WIKIDATA_API_URL, params=params)
    entities = data.get("entities", {}) if isinstance(data, dict) else {}
    entity = entities.get(entity_id.strip()) if isinstance(entities, dict) else None

    if not entity or entity.get("missing") == "":
        print(f"Entity '{entity_id}' not found or missing.")
        return None

    return entity

def get_entity_details(entity_id, language="en"):
    """
    Fetch and display key entity details in readable form.
    Returns a structured dict for optional downstream use.
    """
    entity = _get_entity_raw(entity_id, language=language)
    if entity is None:
        return None

    eid = entity.get("id", entity_id)
    label = _extract_lang_value(entity.get("labels", {}), language=language)
    description = _extract_lang_value(entity.get("descriptions", {}), language=language)
    aliases = _extract_aliases(entity.get("aliases", {}), language=language)

    sitelinks = entity.get("sitelinks", {}) if isinstance(entity.get("sitelinks"), dict) else {}
    claims = entity.get("claims", {}) if isinstance(entity.get("claims"), dict) else {}

    print("=" * 100)
    print(f"Entity ID   : {eid}")
    print(f"Label       : {label if label else '(missing)'}")
    print(f"Description : {description if description else '(missing)'}")
    print(f"Aliases     : {', '.join(aliases) if aliases else '(none)'}")
    print(f"Sitelinks   : {len(sitelinks)}")
    print(f"Properties with claims: {len(claims)}")
    print("=" * 100)

    # Display sitelinks in tabular form for readability.
    if sitelinks:
        site_rows = []
        for site, info in sitelinks.items():
            if not isinstance(info, dict):
                continue
            site_rows.append({
                "site": site,
                "title": info.get("title", ""),
                "url": info.get("url", ""),
            })
        if site_rows:
            print("\nSitelinks (first 20):")
            display(pd.DataFrame(site_rows).head(20))
    else:
        print("\nSitelinks: (none)")

    # Display simplified claims grouped by property ID.
    print("\nClaims grouped by property (simplified values):")
    if not claims:
        print("  (no claims)")
    else:
        for prop_id, claim_list in claims.items():
            print(f"\n  {prop_id}:")
            if not isinstance(claim_list, list) or not claim_list:
                print("    - (none)")
                continue
            for claim in claim_list[:10]:
                print(f"    - {_decode_claim_value(claim)}")
            if len(claim_list) > 10:
                print(f"    ... ({len(claim_list) - 10} more values)")

    return {
        "id": eid,
        "label": label,
        "description": description,
        "aliases": aliases,
        "sitelinks": sitelinks,
        "claims": claims,
    }

## Helper Functions
Convenience wrappers for common interactive exploration tasks.

In [4]:
# Helper functions requested in the specification.

def print_entity_summary(entity_id, language="en"):
    """Print a compact summary: ID, label, description, and counts."""
    entity = _get_entity_raw(entity_id, language=language)
    if entity is None:
        return

    label = _extract_lang_value(entity.get("labels", {}), language=language)
    description = _extract_lang_value(entity.get("descriptions", {}), language=language)
    aliases = _extract_aliases(entity.get("aliases", {}), language=language)
    sitelinks = entity.get("sitelinks", {}) if isinstance(entity.get("sitelinks"), dict) else {}
    claims = entity.get("claims", {}) if isinstance(entity.get("claims"), dict) else {}

    print("=" * 80)
    print(f"ID          : {entity.get('id', entity_id)}")
    print(f"Label       : {label if label else '(missing)'}")
    print(f"Description : {description if description else '(missing)'}")
    print(f"Alias count : {len(aliases)}")
    print(f"Sitelinks   : {len(sitelinks)}")
    print(f"Claim props : {len(claims)}")
    print("=" * 80)

def show_aliases(entity_id, language="en"):
    """Show aliases for an entity in selected language (with fallback)."""
    entity = _get_entity_raw(entity_id, language=language)
    if entity is None:
        return

    aliases = _extract_aliases(entity.get("aliases", {}), language=language)
    print(f"Aliases for {entity.get('id', entity_id)} ({language}):")
    if not aliases:
        print("  (none)")
        return

    for a in aliases:
        print(f"  - {a}")

def show_claims(entity_id, language="en", max_properties=20, max_values_per_property=5):
    """Show simplified claims grouped by property, with configurable truncation."""
    entity = _get_entity_raw(entity_id, language=language)
    if entity is None:
        return

    claims = entity.get("claims", {}) if isinstance(entity.get("claims"), dict) else {}
    if not claims:
        print("No claims found.")
        return

    prop_ids = list(claims.keys())
    print(f"Showing up to {max_properties} properties for {entity_id}:")

    for prop_id in prop_ids[:max_properties]:
        values = claims.get(prop_id, [])
        print(f"\n{prop_id}:")

        if not isinstance(values, list) or not values:
            print("  - (none)")
            continue

        for claim in values[:max_values_per_property]:
            print(f"  - {_decode_claim_value(claim)}")

        if len(values) > max_values_per_property:
            print(f"  ... ({len(values) - max_values_per_property} more values)")

    if len(prop_ids) > max_properties:
        print(f"\n... ({len(prop_ids) - max_properties} more properties not shown)")

## Optional: Wikidata Query Service (SPARQL)
This function runs SPARQL queries against `https://query.wikidata.org/sparql` and returns a DataFrame.

In [5]:
# Optional SPARQL support for structured graph queries.

def run_wikidata_sparql(query, timeout=60):
    """Run SPARQL query on WDQS and return results as a DataFrame."""
    if not isinstance(query, str) or not query.strip():
        print("Please provide a non-empty SPARQL query string.")
        return pd.DataFrame()

    headers = {
        "Accept": "application/sparql-results+json",
        "User-Agent": DEFAULT_HEADERS["User-Agent"],
    }

    try:
        response = requests.get(
            WDQS_URL,
            params={"query": query, "format": "json"},
            headers=headers,
            timeout=timeout,
        )
        response.raise_for_status()
        data = response.json()
    except requests.RequestException as exc:
        print(f"SPARQL request failed: {exc}")
        return pd.DataFrame()
    except ValueError as exc:
        print(f"Invalid SPARQL JSON response: {exc}")
        return pd.DataFrame()

    bindings = data.get("results", {}).get("bindings", [])
    if not bindings:
        print("SPARQL returned no rows.")
        return pd.DataFrame()

    rows = []
    for b in bindings:
        row = {}
        for var, payload in b.items():
            if isinstance(payload, dict):
                row[var] = payload.get("value", "")
            else:
                row[var] = str(payload)
        rows.append(row)

    return pd.DataFrame(rows)

## Demos: Search Terms
Try these terms and inspect candidate entities.

In [6]:
# Demo 1: Apple
df_apple = search_wikidata("Apple", limit=10)
display(df_apple)

,id,label,description,match_text,aliases,concepturi
0,Q312,Apple Inc.,American multinational technology company base...,Apple Inc.,,http://www.wikidata.org/entity/Q312
1,Q213710,Apple Records,UK international record label; imprint of Appl...,Apple Records,,http://www.wikidata.org/entity/Q213710
2,Q89,apple,edible fruit of the apple tree,apple,,http://www.wikidata.org/entity/Q89
3,Q20056642,Apple Music,Internet online music service by Apple,Apple Music,,http://www.wikidata.org/entity/Q20056642
4,Q26944931,Apple,unisex given name,Apple,,http://www.wikidata.org/entity/Q26944931
5,Q111046003,Apple,registered trademark owned by Apple,Apple,,http://www.wikidata.org/entity/Q111046003
6,Q104819,Malus,flowering genus in the rose family Rosaceae,apples,apples,http://www.wikidata.org/entity/Q104819
7,Q9589,iTunes,media player and library software,Apple iTunes,Apple iTunes,http://www.wikidata.org/entity/Q9589
8,Q158657,Malus pumila,species of apple tree,apple,apple,http://www.wikidata.org/entity/Q158657
9,Q62446736,Apple TV,video streaming service by Apple Inc.,Apple TV,,http://www.wikidata.org/entity/Q62446736


In [7]:
# Demo 2: United States
df_us = search_wikidata("United States", limit=10)
display(df_us)

,id,label,description,match_text,aliases,concepturi
0,Q30,United States,country located primarily in North America,United States,,http://www.wikidata.org/entity/Q30
1,Q637413,United States Census Bureau,U.S. agency responsible for the census and rel...,United States Census Bureau,,http://www.wikidata.org/entity/Q637413
2,Q96,Mexico,country in North America,United States of Mexico,United States of Mexico,http://www.wikidata.org/entity/Q96
3,Q11703,United States Virgin Islands,unincorporated territory of the United States ...,United States Virgin Islands,,http://www.wikidata.org/entity/Q11703
4,Q4917,United States dollar,official currency of the United States,United States dollar,,http://www.wikidata.org/entity/Q4917
5,Q214867,National Gallery of Art,"national art museum in Washington, D.C.",United States. National Gallery of Art,United States. National Gallery of Art,http://www.wikidata.org/entity/Q214867
6,Q29468,Republican Party,American political party,United States Republican Party,United States Republican Party,http://www.wikidata.org/entity/Q29468
7,Q29552,Democratic Party,American political party,United States Democratic Party,United States Democratic Party,http://www.wikidata.org/entity/Q29552
8,Q60346,National Institute for Occupational Safety and...,United States government research agency for p...,United States Bureau of Occupational Safety an...,United States Bureau of Occupational Safety an...,http://www.wikidata.org/entity/Q60346
9,Q11201,Supreme Court of the United States,highest court of jurisdiction in the United St...,United States Supreme Court,United States Supreme Court,http://www.wikidata.org/entity/Q11201


In [ ]:
# Demo 3: director
df_director = search_wikidata("director", limit=10)
display(df_director)

In [8]:
# Demo 4: OpenAI
df_openai = search_wikidata("OpenAI", limit=10)
display(df_openai)

,id,label,description,match_text,aliases,concepturi
0,Q21708200,OpenAI,American artificial intelligence research orga...,OpenAI,,http://www.wikidata.org/entity/Q21708200
1,Q96237091,OpenAI API,"various AI APIs, products of OpenAI",OpenAI API,,http://www.wikidata.org/entity/Q96237091
2,Q25106053,OpenAIRE,"network of Open Access repositories, archives ...",OpenAIRE,,http://www.wikidata.org/entity/Q25106053
3,Q124544998,Sora,text-to-video model developed by OpenAI,OpenAI Sora,OpenAI Sora,http://www.wikidata.org/entity/Q124544998
4,Q124605186,OpenAI,"artificial intelligence company, onwer and ope...",OpenAI,,http://www.wikidata.org/entity/Q124605186
5,Q130288245,OpenAI o1,2024 AI model from OpenAI,OpenAI o1,,http://www.wikidata.org/entity/Q130288245
6,Q131535482,OpenAI o3,large language model developed by OpenAI,OpenAI o3,,http://www.wikidata.org/entity/Q131535482
7,Q134303330,OpenAI o4-mini,reasoning language model,OpenAI o4-mini,,http://www.wikidata.org/entity/Q134303330
8,Q120549620,python-openai,Python library,openai,openai,http://www.wikidata.org/entity/Q120549620
9,Q116374474,OpenAI OpCo,"artificial intelligence company, for-profit su...",OpenAI OpCo,,http://www.wikidata.org/entity/Q116374474


## Demos: Inspect One Entity in Depth
Set any entity ID you want to inspect below.

Tip: pick an ID from the search results above (for example, often `Q30` for United States and `Q24210` for Apple Inc.).

In [ ]:
# Pick an entity ID and inspect details.
entity_id = "Q30"  # Change this interactively

print_entity_summary(entity_id)
show_aliases(entity_id)
show_claims(entity_id, max_properties=15, max_values_per_property=5)

# Full detail display
_ = get_entity_details(entity_id)

In [ ]:
# Optional SPARQL demo: retrieve a few direct properties for one entity.
# Example uses Q30 (United States). Replace with another entity ID if desired.

selected_entity = "Q30"
sparql_query = f"""
SELECT ?prop ?propLabel ?value ?valueLabel WHERE {{
  wd:{selected_entity} ?p ?value .
  ?prop wikibase:directClaim ?p .
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
}}
LIMIT 25
"""

df_sparql = run_wikidata_sparql(sparql_query)
display(df_sparql)

## Why Wikidata Is Useful
Wikidata works especially well for:
- **Named entities** (people, organizations, places, products, events)
- **Aliases and variant names** (alternate spellings, abbreviations, multilingual names)
- **Entity disambiguation** (same surface word, different meanings/entities)
- **Structured knowledge** (properties and typed values that are machine-readable)

This makes Wikidata a strong resource for tasks like entity linking, search enrichment, abbreviation expansion, and building knowledge-aware NLP pipelines.